# 空间维度分析

In [ ]:
%reload_ext autoreload
%autoreload 2 
# 上面这俩用于自动重载

import sqlite3 # 先导入内置的模块

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, quality, viz
from signal_insight.viz import (MAIN,MAIN_LIGHT,MAIN_DARK,MAIN_MID,GRAY,
                    GRAY_DARK,RED,AMBER,INK,MUTED,FAINT) # 配色常量

viz.apply_theme(theme="default")

con = sqlite3.connect(config.DB_PATH) # 连接数据库
DAY = config.ANALYSIS_DAY
NB = "D03_信令数据洞察"
SOURCE = f"{config.SOURCE_NOTE}:脚本:{NB} 制图:{config.AUTHOR}"

## 基站表初识：扇区和站点

需求：基站表有多少行，编号唯一吗？坐标有没有缺的

In [ ]:
# 新增七张表：先各做一个数据画像，看下基本信息
# 新拿到的 7 张静态表，先各做一张画像：行数、列数、哪些字段有缺失
static = ["ods_cell", "ods_poi_purpose", "ods_poi_transport",
          "ods_aoi_v1", "ods_aoi_v2", "ods_aoi_json",
          "ods_district_boundary"]
rows = []
for t in static:
    p = quality.profile(pd.read_sql(f"SELECT * FROM {t}", con))
    gaps = p[p["缺失数"] > 0]
    rows.append({"表": t, "列数": len(p), "有缺失的字段": "、".join(f"{f}（{n}）" for f, n in zip(gaps["字段"], gaps["缺失数"])) or "无"})

在 GIS 里，**POI 和 AOI 本质上是空间数据模型中的“点要素”和“面要素”**，只是语义不同。

| 维度 | POI | AOI |
|---|---|---|
| 全称 | Point of Interest | Area of Interest |
| 中文 | 兴趣点 | 兴趣面 / 关注区域 |
| GIS 几何 | 点 Point | 面 Polygon / MultiPolygon |
| 维度 | 0 维 | 2 维 |
| 语义 | 一个具体地点、设施、店铺、站点 | 一个区域、范围、边界、围栏 |
| 属性 | 名称、类别、地址、坐标、电话等 | 名称、类型、面积、周长、父级区域等 |
| 典型操作 | 搜索、定位、最近邻、导航终点 | 裁剪、统计、叠加、围栏、区域聚合 |


在 GIS 里，POI 和 AOI 的核心关系是**点—面关系**：

- `ST_Contains(AOI, POI)`：AOI 是否包含 POI
- `ST_Within(POI, AOI)`：POI 是否落在 AOI 内
- `ST_Intersects(AOI, POI)`：是否相交
- 空间连接：把 POI 的属性挂到所属 AOI 上
- 聚合统计：统计每个 AOI 内有多少餐饮、便利店、地铁站
- 密度分析：POI 核密度、点密度
- 缓冲区分析：围绕 POI 或 AOI 生成影响范围
- 地理围栏：用 AOI 判断设备、车辆、人员是否进入/离开


> **POI 是点抽象，用于定位和检索；AOI 是面抽象，用于范围和分析。**

一句话总结：  
**POI 是 GIS 中的点要素，AOI 是 GIS 中的面要素；AOI 可包含多个 POI，POI 可通过空间连接归属到 AOI，二者共同支撑搜索、统计、规划和地理围栏等空间分析。**

In [ ]:
cell = pd.read_sql("SELECT * FROM ods_cell", con)
from signal_insight.quality import profile
profile(cell)

In [ ]:
# 没坐标点扇区定位不了
cell = cell.dropna(subset=['lng','lat']).copy()
# 不重复的坐标=一个站点
sites = cell[['lng', 'lat']].drop_duplicates().reset_index(drop=True)
# 给站点做编号：编号补0到S00005
sites['site_id'] = "S" + (sites.index + 1).astype(str).str.zfill(5)
# 把站点号贴回每个扇区
cell = cell.merge(sites, on=['lng', 'lat'],how='left')

In [ ]:
per_site = cell.groupby('site_id').size()
sector_dist = per_site.value_counts().sort_index() \
    .rename_axis('扇区数').reset_index(name='站点数')
sector_dist['站点占比'] = sector_dist['站点数'] / sector_dist['站点数'].sum()

In [ ]:
sector_dist.to_sql("ads_sectors_per_site",
            con,
            if_exists='replace',
            index=False)

In [ ]:
from matplotlib.ticker import FuncFormatter
sec = sector_dist
SOURCE = "数据来源：沈阳移动基站工参（脱敏）；脚本：站点扇区数分布；配置 <版本号+哈希>"
TAIL_K = 9        # 大于这个数的合并成一组单独标记
OUTLIER_K = 20    # 大于这个数的站点需要核查


def fmt_pct(p, digits=0):
    """第 6 节：一般取整；很小的值自动多保留位数（0.01%，不写 0%）。"""
    if p == 0:
        return "0%"
    while round(p * 100, digits) == 0 and digits < 4:
        digits += 1
    return f"{p:.{digits}%}"


def bar_pct(p):
    """条形标注：≥10% 取整，<10% 的数彼此接近，保留一位。"""
    return fmt_pct(p, 0 if p >= 0.1 else 1)


# %% 1 指标（全部由代码算，标题里的数字跟着数据走）
sec = sec[["扇区数", "站点数"]].astype(int).sort_values("扇区数").reset_index(drop=True)
n_site = sec["站点数"].sum()
share = sec["站点数"] / n_site

mode_k = int(sec.loc[sec["站点数"].idxmax(), "扇区数"])
mode_p = sec["站点数"].max() / n_site
MULT3 = [k for k in (3, 6, 9) if k <= TAIL_K]
mult3_p = sec.loc[sec["扇区数"].isin(MULT3), "站点数"].sum() / n_site
one_p = sec.loc[sec["扇区数"] == 1, "站点数"].sum() / n_site
median_k = int(sec.loc[share.cumsum() >= 0.5, "扇区数"].iloc[0])
mean_k = (sec["扇区数"] * sec["站点数"]).sum() / n_site

# 大于 TAIL_K 的一组，再拆成三段写在标注里
tail = sec[sec["扇区数"] > TAIL_K]
n_tail, max_k = int(tail["站点数"].sum()), int(sec["扇区数"].max())
seg = {
    f"{TAIL_K + 1}–12": tail.loc[tail["扇区数"] <= 12, "站点数"].sum(),
    f"13–{OUTLIER_K}": tail.loc[tail["扇区数"].between(13, OUTLIER_K), "站点数"].sum(),
    f">{OUTLIER_K}": tail.loc[tail["扇区数"] > OUTLIER_K, "站点数"].sum(),
}
n_out = int(seg[f">{OUTLIER_K}"])


# %% 2 分组：1–TAIL_K 逐个显示，大于 TAIL_K 合并为一组
main = sec.set_index("扇区数")["站点数"].reindex(range(1, TAIL_K + 1), fill_value=0)
labels = [str(k) for k in range(1, TAIL_K + 1)] + [f">{TAIL_K}"]
values = list(main.to_numpy()) + [n_tail]
xs = list(range(TAIL_K)) + [TAIL_K + 0.6]                  # 合并组与前面隔开
colors = [MAIN if k in MULT3 else GRAY for k in range(1, TAIL_K + 1)] + [GRAY_DARK]

# %% 3 出图：纵向柱状（离散分布），纵轴从 0 开始
fig, ax = plt.subplots(figsize=(10, 5.6), dpi=100)
ax.bar(xs, values, width=0.72, color=colors)

ymax = max(values)
ax.set_ylim(0, ymax * 1.18)
ax.set_xlim(-0.7, xs[-1] + 0.7)
ax.set_xticks(xs)
ax.set_xticklabels(labels)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set_xlabel("每个站点的扇区数")
ax.set_ylabel("站点数")
ax.grid(axis="y", color=GRAY, lw=0.6, alpha=0.6)
ax.set_axisbelow(True)

# 柱顶写 站点数 + 占比
for x, v, lab in zip(xs, values, labels):
    ax.text(x, v + ymax * 0.012, f"{v:,}\n{bar_pct(v / n_site)}",
            ha="center", va="bottom", fontsize=9, linespacing=1.15,
            color=MAIN_DARK if lab in map(str, MULT3) else INK)

# 合并组：虚线隔开 + 拆分明细，>OUTLIER_K 的一行用琥珀色（警告：需核查）
ax.axvline(TAIL_K - 0.2, color=GRAY_DARK, lw=0.8, ls="--")
x_note, y_note = TAIL_K - 0.45, ymax * 0.70          # 放在虚线左侧的空白处
ax.text(x_note, y_note, f">{TAIL_K} 个扇区的 {n_tail:,} 个站点合并为右侧一组：",
        ha="right", va="bottom", fontsize=9.5, color=INK, fontweight="bold")
lines = [(f"{k} 个扇区：{int(v):,} 个站点", MUTED) for k, v in seg.items()]
lines[-1] = (f">{OUTLIER_K} 个扇区：{n_out} 个站点，最多 {max_k} 个，需核查", AMBER)
for i, (txt, c) in enumerate(lines):
    ax.text(x_note, y_note - (i + 1) * ymax * 0.055, txt,
            ha="right", va="bottom", fontsize=9.5, color=c)
ax.annotate("", xy=(xs[-1], n_tail + ymax * 0.12), xytext=(x_note + 0.1, y_note + ymax * 0.02),
            arrowprops=dict(arrowstyle="-", color=GRAY_DARK, lw=0.8,
                            connectionstyle="angle,angleA=0,angleB=90,rad=0"))

ax.text(4.3, ymax * 0.86,
        f"深色 = 3 的倍数，合计 {fmt_pct(mult3_p)}\n"
        "多为三扇区 × 多频段或多载波叠加（推测，待用频段字段核实）",
        ha="center", va="center", fontsize=9.5, color=MAIN_DARK)

title = (f"{mode_k} 扇区站点最多，占 {fmt_pct(mode_p)}；"
         f"{'/'.join(map(str, MULT3))} 扇区合计占 {fmt_pct(mult3_p)}")
subtitle = (f"共 {n_site:,} 个站点；扇区数 = 同一站点下不同扇区（小区）的个数，"
            f"中位 {median_k} 个、平均 {mean_k:.1f} 个"
            f"\n单扇区站点占 {fmt_pct(one_p)}，只能定位到站点、没有方向信息；"
            f"大于 {TAIL_K} 个的合并为一组")
assert len(title) <= 40, len(title)
viz.finish(fig, title, subtitle, SOURCE, "Q03_站点扇区数分布.png")

In [ ]:
from signal_insight import quality
quality.dq_record(con,
      "DQ-10",
      "完整性",
      "基站坐标缺失",
      "ods_cell.lng / lat",
      "6 行",
      "通过",
      evidence=f"共 {len(cell) + 6:,} 行，缺坐标的占比不到 0.02%",
      decision="删除这 6 行")

## 需求：信令能不能关联上基站表

需求：dwd_signal里面的每条记录，能不能在基站表里找到坐标

In [ ]:
# 读取dwd表
dwd = pd.read_sql("SELECT * FROM dwd_signal", con, parse_dates=['ts'])
# 两个表格合并
joined = dwd.merge(cell[['lac_cell', "site_id", 'lng', 'lat']],
                on='lac_cell',
                how='left')

rec_rate = joined['lng'].notna().mean()
cells_used = dwd['lac_cell'].nunique()

miss = joined.loc[joined['lng'].isna(), "lac_cell"].drop_duplicates()
cell_missing = len(miss)
no_id = (~miss.isin(
    pd.read_sql("SELECT lac_cell FROM ods_cell", con)['lac_cell'])).sum()

- 就是按记录，99.9984%能关联上坐标，对不上的有8条，按扇区，信令我们一共用到了19,787个扇区.8个关联不到坐标，2个基站表里面没有编号，6个有编号但是缺坐标

In [ ]:
quality.dq_record(
con,
"DQ-11",
"一致性",
"信令与基站表的关联率",
"dwd_signal.lac_cell <-> ods_cell.lac_cell",
f"按记录 {rec_rate:.3%}",
"通过",
evidence=
f"信令用到 {cells_used:,} 个扇区，{cell_missing} \
个关联不到坐标（{no_id} 个基站表里没有编号，其余缺坐标）",
decision="关联不上的记录在空间分析中剔除")

## 点落面：每个站点在什么区

In [ ]:
from matplotlib.path import Path as Mppath


def rings_from_points(points, key):
    """从一行一个顶点的表来还原多边形：{名称:[环1, 环2]}每个都是N X 2的经纬度数组"""
    pts = points if "ring" in points.columns else points.assign(ring=0)
    return {
        name: [
            r.sort_values("seq")[['lng', 'lat']].to_numpy(dtype=float)
            for _, r in g.groupby("ring")
        ]
        for name, g in pts.groupby(key, sort=False)
    }


def point_in_ploygons(lng, lat, polys):
    """点落面：返回每个点所在的多边形区域，不在任何多边形里面的返回None"""
    xy = np.column_stack(
        [np.asarray(lng, dtype=float),
         np.asarray(lat, dtype=float)])
    out = np.full(len(xy), None, dtype=object)
    for name, rings in polys.items():
        inside = np.zeros(len(xy), dtype=bool)
        for ring in rings:
            # ^= 是异或域：被奇数个环包住 = 在里面，被偶数个环包住 = 在洞里
            inside ^= Mppath(ring).contains_points(xy)
        out[inside & pd.isna(out)] = name
    return out

# 还原
districts = rings_from_points(
    pd.read_sql("SELECT * FROM ods_district_boundary", con), "district")

sites['districts'] = point_in_ploygons(sites['lng'], sites['lat'], districts)
sites['districts'] = sites['districts'].fillna("市外")

In [ ]:
sites

In [ ]:
out_share = (sites['districts'] == '市外').mean()
rec_district = joined.merge(sites[['site_id', 'districts']],
            on="site_id",how="left")
rec_out = (rec_district['districts'] =='市外').mean()

```markdown
我手里有这些数据：
print(
    pd.read_sql("SELECT * FROM ods_district_boundary",
                con).head(10).to_markdown())
|    |   adcode | district   |   ring |   seq |     lng |     lat |
|---:|---------:|:-----------|-------:|------:|--------:|--------:|
|  0 |   210102 | 和平区     |      0 |     0 | 123.429 | 41.8174 |
|  1 |   210102 | 和平区     |      0 |     1 | 123.43  | 41.8109 |
|  2 |   210102 | 和平区     |      0 |     2 | 123.423 | 41.7899 |
|  3 |   210102 | 和平区     |      0 |     3 | 123.425 | 41.7893 |
|  4 |   210102 | 和平区     |      0 |     4 | 123.428 | 41.7782 |
|  5 |   210102 | 和平区     |      0 |     5 | 123.436 | 41.7782 |
|  6 |   210102 | 和平区     |      0 |     6 | 123.435 | 41.7591 |
|  7 |   210102 | 和平区     |      0 |     7 | 123.439 | 41.7538 |
|  8 |   210102 | 和平区     |      0 |     8 | 123.446 | 41.7484 |
|  9 |   210102 | 和平区     |      0 |     9 | 123.441 | 41.7448 |
```
利用这些函数处理做了如下判断
```python
from matplotlib.path import Path as Mppath


def rings_from_points(points, key):
    """从一行一个顶点的表来还原多边形：{名称:[环1, 环2]}每个都是N X 2的经纬度数组"""
    pts = points if "ring" in points.columns else points.assign(ring=0)
    return {
        name: [
            r.sort_values("seq")[['lng', 'lat']].to_numpy(dtype=float)
            for _, r in g.groupby("ring")
        ]
        for name, g in pts.groupby(key, sort=False)
    }


def point_in_ploygons(lng, lat, polys):
    """点落面：返回每个点所在的多边形区域，不在任何多边形里面的返回None"""
    xy = np.column_stack(
        [np.asarray(lng, dtype=float),
         np.asarray(lat, dtype=float)])
    out = np.full(len(xy), None, dtype=object)
    for name, rings in polys.items():
        inside = np.zeros(len(xy), dtype=bool)
        for ring in rings:
            # ^= 是异或域：被奇数个环包住 = 在里面，被偶数个环包住 = 在洞里
            inside ^= Mppath(ring).contains_points(xy)
        out[inside & pd.isna(out)] = name
    return out

# 还原
districts = rings_from_points(
    pd.read_sql("SELECT * FROM ods_district_boundary", con), "district")

sites['districts'] = point_in_ploygons(sites['lng'], sites['lat'], districts)
sites['districts'] = sites['districts'].fillna("市外")

sites['districts'].value_counts()


out_share = (sites['districts'] == '市外').mean()
rec_district = joined.merge(sites[['site_id', 'districts']],
            on="site_id",how="left")
rec_out = (rec_district['districts'] =='市外').mean()

districts
市外      4808
铁西区     1129
浑南区      942
沈河区      878
和平区      831
大东区      788
沈北新区     741
于洪区      718
皇姑区      709
新民市      622
苏家屯区     565
法库县      496
辽中区      441
康平县      438
Name: count, dtype: int64
```
sites表里面有lng, lat, site_id和districts信息，


In [ ]:
from itertools import combinations

import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.path import Path


def scale_bar(ax, km):
    """比例尺：按纬度把公里换成经度差。"""
    (x0_, x1_), (y0_, y1_) = ax.get_xlim(), ax.get_ylim()
    deg = km / (111.32 * np.cos(np.radians(lat0)))
    xs, ys = x0_ + 0.04 * (x1_ - x0_), y0_ + 0.04 * (y1_ - y0_)
    ax.plot([xs, xs + deg], [ys, ys], color=INK, lw=1.2, solid_capstyle="butt")
    for x in (xs, xs + deg):                                         # 两端短竖线
        ax.plot([x, x], [ys, ys + 0.008 * (y1_ - y0_)], color=INK, lw=0.8)
    ax.text(xs + deg / 2, ys + 0.018 * (y1_ - y0_), f"{km} 公里",
            ha="center", va="bottom", fontsize=9, color=INK)


def north_arrow(ax, x=0.95, y=0.95):
    ax.annotate("北", xy=(x, y), xytext=(x, y - 0.08), xycoords="axes fraction",
                ha="center", va="top", fontsize=9, color=INK,
                arrowprops=dict(arrowstyle="-|>", color=INK, lw=0.8, shrinkA=0, shrinkB=0))


def draw_districts(ax, polys, values=None, cmap=None, norm=None, edge="#9AA3AE", lw=0.6):
    """区县边界；给了 values 就按数值填色（分级设色地图）。多边形由调用方传进来，函数里不读库。
    返回所有顶点的范围 (经度最小, 经度最大, 纬度最小, 纬度最大)，用来设视窗。"""
    for name, rings in polys.items():
        fc = cmap(norm(values[name])) if values is not None and name in values else "none"
        for ring in rings:
            ax.fill(ring[:, 0], ring[:, 1], facecolor=fc, edgecolor=edge, lw=lw)
    allpts = np.vstack([r for rings in polys.values() for r in rings])
    return (allpts[:, 0].min(), allpts[:, 0].max(),
            allpts[:, 1].min(), allpts[:, 1].max())


lat0 = 41.8                                          # 沈阳大约的纬度，用来校正地图长宽比
LINE = dict(color=MUTED, lw=0.45)                    # 引线：细、灰，不和区界、蓝点抢

# ---------------- 区县标注：落点 / 分侧 / 拐点（D3_02、D3_03 共用，两张图连线一致）----------------
ELBOW, STUB, TXT_GAP = 0.07, 0.14, 0.02      # 度：拐点离地图外缘、等长短横线、文字离短横线
MAP_H_IN, LABEL_W_IN, HEAD_IN = 7.0, 1.30, 0.50   # 英寸：地图高度、标注文字预留宽度、右图列标题行高。D3_02 / D3_03 必须一致，地图才一样大


def map_frame(bx0, bx1, by0, by1, lat0):
    """由地图外框反推：子图纵向范围、子图宽度（英寸）、横向范围。
    同一份边界 + 同一个 MAP_H_IN → 两张图里的地图一样大，位置也一样。"""
    y0, y1 = by0 - 0.03, by1 + 0.03                                    # 上下各留 0.03°
    dy = (y1 - y0) / MAP_H_IN                                          # 每英寸多少纬度
    dx = dy / np.cos(np.radians(lat0))                                 # 每英寸多少经度
    side_in = (ELBOW + STUB + TXT_GAP) / dx + LABEL_W_IN + 0.05        # 拐点、短横线、文字间隙，再加文字宽度
    w_in = (bx1 - bx0) / dx + 2 * side_in
    xc = (bx0 + bx1) / 2
    return (y0, y1), w_in, (xc - w_in / 2 * dx, xc + w_in / 2 * dx)


def draw_outline(ax, polys, color=None, lw=0.9):
    """全市外缘：只出现一次的边就是外缘（两个区共用的边出现两次，不画），画一圈深色细线。
    区与区之间用白线分开（draw_districts 里 edge="white", lw=0.9）。"""
    n_seen, segs = {}, []
    for rings in polys.values():
        for r in rings:
            for p_, q_ in zip(r, np.roll(r, -1, axis=0)):
                if np.allclose(p_, q_):
                    continue
                key = frozenset((tuple(np.round(p_, 5)), tuple(np.round(q_, 5))))
                n_seen[key] = n_seen.get(key, 0) + 1
                segs.append((key, (p_, q_)))
    ax.add_collection(LineCollection([sg for k_, sg in segs if n_seen[k_] == 1],
                                     colors=color or INK, linewidths=lw, capstyle="round", zorder=3))


def pole_point(rings, lat0):
    """落点：区内离边界最远的点。一定落在区里、不贴边，引线进区的位置比"离中心最近的站点"稳。"""
    r = max(rings, key=len)
    lo, hi = r.min(0), r.max(0)
    gx, gy = np.meshgrid(np.linspace(lo[0], hi[0], 45), np.linspace(lo[1], hi[1], 45))
    g = np.column_stack([gx.ravel(), gy.ravel()])
    g = g[Path(r).contains_points(g)]
    if len(g) == 0:
        return r.mean(axis=0)
    edge = np.vstack([np.linspace(r[i], r[(i + 1) % len(r)], 6, endpoint=False) for i in range(len(r))])
    edge = edge[:: max(1, len(edge) // 400)]
    k = [np.cos(np.radians(lat0)), 1]                                  # 经度差按纬度折成同一长度
    dist = np.sqrt((((g[:, None, :] - edge[None, :, :]) * k) ** 2).sum(-1)).min(1)
    return g[dist.argmax()]


def seg_cross(p1, p2, p3, p4):
    ccw = lambda a, b, c: (c[1] - a[1]) * (b[0] - a[0]) - (b[1] - a[1]) * (c[0] - a[0])
    return ccw(p1, p3, p4) * ccw(p2, p3, p4) < 0 and ccw(p1, p2, p3) * ccw(p1, p2, p4) < 0


def label_districts(ax, polys, texts, bx, ylim, lat0, hot=(), fs=9):
    """左右两列上下等距标注。引线 = 等长短横线 → 拐点 → 直线进区。
    polys {区名: [环]}；texts {区名: 标注文字}；bx=(地图最西, 最东)；ylim=(子图下沿, 上沿)；hot 里的区名用 MAIN 加粗。
    分侧：落点按经度排序，西边一半在左列、东边一半在右列。
    拐点：贴着地图外缘，同一列在同一条竖线上，所以进区的斜线从同一处散开，不会各走各的。
    行位：两列共用同一个上下范围；同一列内引线若交叉就对调位置，直到不交叉。"""
    y0, y1 = ylim
    span = y1 - y0
    anchor = {n: pole_point(r, lat0) for n, r in polys.items()}
    top = min(max(a[1] for a in anchor.values()) + 0.04 * span, y1 - 0.2 * span)   # 顶上留给指北针
    bot = max(min(a[1] for a in anchor.values()) - 0.04 * span, y0 + 0.1 * span)   # 底下留给比例尺
    by_x = sorted(anchor, key=lambda n: anchor[n][0])
    half = (len(by_x) + 1) // 2
    for left, names in ((True, by_x[:half]), (False, by_x[half:])):
        ex = bx[0] - ELBOW if left else bx[1] + ELBOW                  # 拐点
        tx = ex - STUB if left else ex + STUB                          # 文字端
        order = sorted(names, key=lambda n: -anchor[n][1])             # 北 → 南
        ty = dict(zip(order, np.linspace(top, bot, len(order))))
        for _ in range(500):
            moved = False
            for a, b in combinations(order, 2):
                if seg_cross((ex, ty[a]), anchor[a], (ex, ty[b]), anchor[b]):
                    ty[a], ty[b] = ty[b], ty[a]
                    moved = True
            if not moved:
                break
        for n in order:
            lng, lat = anchor[n]
            ax.plot([tx, ex, lng], [ty[n], ty[n], lat], zorder=5, solid_joinstyle="round", **LINE)
            ax.plot(lng, lat, "o", ms=2.2, color=INK, mec="white", mew=0.4, zorder=6)
            ax.text(tx + (-TXT_GAP if left else TXT_GAP), ty[n], texts[n],
                    ha="right" if left else "left", va="center", fontsize=fs, zorder=6,
                    color=MAIN if n in hot else INK, fontweight="bold" if n in hot else "normal")


# ---- 版面：地图大小与 D3_03 一致（共用 MAP_H_IN / map_frame），宽度由地图长宽比反推 ----
allpts = np.vstack([r for rings in districts.values() for r in rings])
bx0, by0 = allpts.min(0)
bx1, by1 = allpts.max(0)
(y0, y1), left_w_in, (ax0, ax1) = map_frame(bx0, bx1, by0, by1, lat0)
RIGHT_W_IN = 4.2
fig_w = 0.3 + left_w_in + 0.35 + RIGHT_W_IN + 0.2
fig_h = MAP_H_IN + 0.67 + 0.23 * 1 + 0.32 + 0.25                  # 1 = 副标题行数（viz.finish 的留白）
fig = plt.figure(figsize=(fig_w, fig_h))
gs = fig.add_gridspec(2, 3, hspace=0, wspace=0, height_ratios=[HEAD_IN, MAP_H_IN - HEAD_IN],
                      width_ratios=[left_w_in, 0.35, RIGHT_W_IN])
a1 = fig.add_subplot(gs[:, 0])
a2 = fig.add_subplot(gs[1, 2])

# ============ 左：市内按站点数填色（单色浅→深），市外画蓝点 ============
counts = sites.loc[sites["districts"] != "市外", "districts"].value_counts()
cmap = LinearSegmentedColormap.from_list("站点数", ["white", GRAY_DARK])
norm = Normalize(0, counts.max())
draw_districts(a1, districts, values=counts, cmap=cmap, norm=norm, edge="white", lw=0.9)   # 白线分区，与 D3_03 一致
x0, x1 = bx0 - 0.15, bx1 + 0.15
view = sites[sites["lng"].between(x0, x1) & sites["lat"].between(y0, y1)]
hidden = len(sites) - len(view)
out = view[view["districts"] == "市外"]
inn = view[view["districts"] != "市外"]
a1.scatter(inn["lng"], inn["lat"], s=1, color=INK, lw=0, zorder=3)     # 市内站点：深色点，叠在填色上
a1.scatter(out["lng"], out["lat"], s=1, color=MAIN, lw=0, zorder=3)    # 市外站点：蓝点
draw_outline(a1, districts)                                         # 全市外缘深色线（与 D3_03 共用），画在散点之后不被盖住
a1.set_xlim(ax0, ax1)
a1.set_ylim(y0, y1)
a1.set_aspect(1 / np.cos(np.radians(lat0)), adjustable="box")      # 长宽比校正；视窗已按此比例算好
a1.set_xticks([]); a1.set_yticks([]); a1.grid(False)
for sp_ in a1.spines.values():
    sp_.set_visible(False)
scale_bar(a1, 20); north_arrow(a1)

# —— 区县标注：落点 / 分侧 / 拐点位置见 label_districts ——
label_districts(a1, districts, {n: f"{n} {counts[n]:,}" for n in districts},
                (bx0, bx1), (y0, y1), lat0)

# ============ 右：两条 100% 堆叠条，同一个“市外”换两个分母 ============
n_out = (sites["districts"] == "市外").sum()
rows = [("按站点", out_share, f"{n_out:,} / {len(sites):,} 个站点"),
        ("按信令", rec_out, "10-03 全天信令记录")]
h = 0.4
for i, (lab, v, note) in enumerate(rows):
    a2.barh(i, v, color=MAIN, height=h)
    a2.barh(i, 1 - v, left=v, color=GRAY, height=h)
    pct = f"{v:.1%}" if v < 0.05 else f"{v:.0%}"
    if v >= 0.15:
        a2.text(v / 2, i, f"市外 {pct}", ha="center", va="center", fontsize=10.5,
                fontweight="bold", color="white")
    else:
        a2.text(v + 0.015, i, f"市外 {pct}", ha="left", va="center", fontsize=10.5,
                fontweight="bold", color=MAIN)
    a2.text(0.985, i, f"市内 {1 - v:.0%}", ha="right", va="center", fontsize=9.5, color=INK)
    a2.text(0, i - h / 2 - 0.07, lab, va="bottom", fontsize=10, fontweight="bold", color=INK)
    a2.text(1, i - h / 2 - 0.07, note, ha="right", va="bottom", fontsize=9, color=MUTED)
a2.fill([0, out_share, rec_out, 0], [h / 2, h / 2, 1 - h / 2, 1 - h / 2],
        color=MAIN_LIGHT, lw=0, zorder=0)                    # 市外那一段从上往下收窄多少
ratio = (rec_out / out_share) / ((1 - rec_out) / (1 - out_share))
a2.text(0, 1 + h / 2 + 0.16,
        f"平均每个市外站点的信令量只有市内站点的 {ratio:.1%}" if ratio < 0.05
        else f"平均每个市外站点的信令量只有市内站点的 {ratio:.0%}",
        va="top", fontsize=10, color=INK)
BAR_UNIT_IN = 1.75                                       # 数据 1 个单位 = 1.75 英寸，图变高时条不会变粗
a2.set_xlim(0, 1); a2.set_ylim(-0.7 + (MAP_H_IN - HEAD_IN) / BAR_UNIT_IN, -0.7)   # 内容靠上，下方留白
a2.set_xticks([]); a2.set_yticks([]); a2.grid(False)
for sp_ in a2.spines.values():
    sp_.set_visible(False)
a2.set_title("市外占比：按站点 vs 按信令", loc="left", fontsize=10.5, color=INK,
             fontweight="bold", pad=4)

title = f"{len(sites):,} 个站点中 {out_share:.0%} 在沈阳市外，但只有 {rec_out:.1%} 的信令发生在那里"
viz.finish(fig, title,
           f"市内按区县站点数填色，越深越多，区名后为站点数；黑点 = 市内站点，蓝点 = 市外站点；"
           f"视窗外另有 {hidden:,} 个站点；口径：dim_cell、dwd_signal（10-03）",
           SOURCE, "D3_02_站点与区县.png")
plt.show()

基站表里的地理范围大于沈阳市市区：34%的站点（4808个）位于市外。但是10月3日仅有1.8%的信令发生的市外站点，所以市外站点予以保留，对市内分析影响较小

In [ ]:
quality.dq_record(con,
          "DQ-12",
          "代表性",
          "基站表的地理范围",
          "ods_cell",
          f"{out_share:.0%} 的站点在沈阳市外",
          "通过",
          evidence=f"只有 {rec_out:.1%} 的信令发生在市外站点",
          decision="保留市外站点，区县标为“市外”")

## 商圈边界信息判定

In [ ]:
v1_pts = pd.read_sql("SELECT * FROM ods_aoi_v1", con)
json_pts = pd.read_sql("SELECT * FROM ods_aoi_json", con)
v2_pts = pd.read_sql("SELECT * FROM ods_aoi_v2", con)

In [ ]:
print(v1_pts.head().to_markdown())

In [ ]:
print(json_pts.head().to_markdown())

In [ ]:
print(v2_pts.head().to_markdown())

### 解决方案提示词
```python
def rings_from_points(points, key):
    """从一行一个顶点的表来还原多边形：{名称:[环1, 环2]}每个都是N X 2的经纬度数组"""
    pts = points if "ring" in points.columns else points.assign(ring=0)
    return {
        name: [
            r.sort_values("seq")[['lng', 'lat']].to_numpy(dtype=float)
            for _, r in g.groupby("ring")
        ]
        for name, g in pts.groupby(key, sort=False)
    }


```
v1_pts内容如下
|    | name       |   seq |     lng |     lat |
|---:|:-----------|------:|--------:|--------:|
|  0 | 太原街商圈 |     0 | 123.41  | 41.8026 |
|  1 | 太原街商圈 |     1 | 123.409 | 41.7964 |
|  2 | 太原街商圈 |     2 | 123.409 | 41.7925 |
|  3 | 太原街商圈 |     3 | 123.409 | 41.7962 |
|  4 | 太原街商圈 |     4 | 123.41  | 41.8026 |

v2_pts内容如下
|    | name       |   seq |     lng |     lat |
|---:|:-----------|------:|--------:|--------:|
|  0 | 太原街商圈 |     0 | 123.41  | 41.8026 |
|  1 | 太原街商圈 |     1 | 123.409 | 41.7964 |
|  2 | 太原街商圈 |     2 | 123.409 | 41.7925 |
|  3 | 太原街商圈 |     3 | 123.409 | 41.7962 |
|  4 | 太原街商圈 |     4 | 123.41  | 41.8026 |

json_pts内容如下：
|    | name       |   seq |           x |           y |
|---:|:-----------|------:|------------:|------------:|
|  0 | 沈阳世博园 |     0 | 1.37641e+07 | 5.11137e+06 |
|  1 | 沈阳世博园 |     1 | 1.37641e+07 | 5.11135e+06 |
|  2 | 沈阳世博园 |     2 | 1.37642e+07 | 5.11134e+06 |
|  3 | 沈阳世博园 |     3 | 1.37642e+07 | 5.11133e+06 |
|  4 | 沈阳世博园 |     4 | 1.37642e+07 | 5.11132e+06 |

其中每个都有好几个区域，主要是希望讲这些区域绘制出来，并排显示，从而决定使用那份数据

### 解决结果

In [ ]:
lat0 = 41.8                                          # 沈阳大约的纬度
KX, KY = 111.32 * np.cos(np.radians(lat0)), 110.57   # 沈阳附近 1 度经度 / 纬度 ≈ 多少公里
R, E = 6378137.0, 0.0818191908426                    # WGS84 长半轴（米）、第一偏心率
LNG_OK, LAT_OK = (122.0, 124.5), (41.0, 43.5)        # 沈阳市域外扩一点：顶点超出就算越界

# ================= 1. 坐标换算：国内三套经纬度 + json 的墨卡托 =================
def wgs2gcj(lng, lat):
    """WGS-84 → GCJ-02（高德、腾讯用的“火星坐标”），公开算法。"""
    a, ee = 6378245.0, 0.00669342162296594323
    x, y = lng - 105.0, lat - 35.0
    dlat = (-100 + 2 * x + 3 * y + 0.2 * y * y + 0.1 * x * y + 0.2 * np.sqrt(abs(x))
            + (20 * np.sin(6 * x * np.pi) + 20 * np.sin(2 * x * np.pi)) * 2 / 3
            + (20 * np.sin(y * np.pi) + 40 * np.sin(y / 3 * np.pi)) * 2 / 3
            + (160 * np.sin(y / 12 * np.pi) + 320 * np.sin(y * np.pi / 30)) * 2 / 3)
    dlng = (300 + x + 2 * y + 0.1 * x * x + 0.1 * x * y + 0.1 * np.sqrt(abs(x))
            + (20 * np.sin(6 * x * np.pi) + 20 * np.sin(2 * x * np.pi)) * 2 / 3
            + (20 * np.sin(x * np.pi) + 40 * np.sin(x / 3 * np.pi)) * 2 / 3
            + (150 * np.sin(x / 12 * np.pi) + 300 * np.sin(x / 30 * np.pi)) * 2 / 3)
    rad = np.radians(lat)
    m = 1 - ee * np.sin(rad) ** 2
    dlat = dlat * 180 / ((a * (1 - ee)) / (m * np.sqrt(m)) * np.pi)
    dlng = dlng * 180 / (a / np.sqrt(m) * np.cos(rad) * np.pi)
    return lng + dlng, lat + dlat


def gcj2wgs(lng, lat):                               # 反算：减去同一位置的偏移（误差在 1 米左右）
    g = wgs2gcj(lng, lat)
    return 2 * lng - g[0], 2 * lat - g[1]


def gcj2bd(lng, lat):
    """GCJ-02 → BD-09（百度）。"""
    z = np.sqrt(lng ** 2 + lat ** 2) + 2e-5 * np.sin(lat * np.pi * 3000 / 180)
    t = np.arctan2(lat, lng) + 3e-6 * np.cos(lng * np.pi * 3000 / 180)
    return z * np.cos(t) + 0.0065, z * np.sin(t) + 0.006


def bd2gcj(lng, lat):
    x, y = lng - 0.0065, lat - 0.006
    z = np.sqrt(x ** 2 + y ** 2) - 2e-5 * np.sin(y * np.pi * 3000 / 180)
    t = np.arctan2(y, x) - 3e-6 * np.cos(x * np.pi * 3000 / 180)
    return z * np.cos(t), z * np.sin(t)


SHIFT = {"无偏移": lambda x, y: (x, y),
         "WGS→GCJ": wgs2gcj, "GCJ→WGS": gcj2wgs, "GCJ→BD": gcj2bd, "BD→GCJ": bd2gcj,
         "WGS→BD": lambda x, y: gcj2bd(*wgs2gcj(x, y)), "BD→WGS": lambda x, y: gcj2wgs(*bd2gcj(x, y))}

# json：平面米坐标 → 经纬度；球面、椭球两种墨卡托都算，后面挑落在沈阳范围里的那种
j_lng = np.degrees(json_pts["x"] / R)
t = np.exp(-json_pts["y"] / R)
phi = np.pi / 2 - 2 * np.arctan(t)
for _ in range(8):
    phi = np.pi / 2 - 2 * np.arctan(t * ((1 - E * np.sin(phi)) / (1 + E * np.sin(phi))) ** (E / 2))
cand = {"EPSG:3857": np.degrees(2 * np.arctan(np.exp(json_pts["y"] / R)) - np.pi / 2),
        "EPSG:3395": np.degrees(phi)}
# 选法：同名区域的顶点中位数，看哪种换算和 v1、v2 离得最近（两种算法南北差 20 多公里，很好分）
ref = pd.concat([v1_pts, v2_pts])
ref = ref[ref["lng"].between(*LNG_OK) & ref["lat"].between(*LAT_OK)].groupby("name")["lat"].median()
jn = json_pts["name"]
proj = min(cand, key=lambda k: np.nanmedian(np.abs(cand[k].groupby(jn).median() - ref.reindex(cand[k].groupby(jn).median().index)))
           if jn.isin(ref.index).any() else abs(np.median(cand[k]) - lat0))

raw = {"v1": v1_pts, "v2": v2_pts, "json": json_pts.assign(lng=j_lng, lat=cand[proj])}
for k in raw:                                         # seq 可能是文本（"10" 会排在 "2" 前面），统一转成数字
    raw[k] = raw[k].assign(seq=pd.to_numeric(raw[k]["seq"], errors="coerce"))

# ================= 2. 顶点级检查：越界、经纬度颠倒、seq 异常 =================
bad_pts = []
for s, d in raw.items():
    out_ = ~(d["lng"].between(*LNG_OK) & d["lat"].between(*LAT_OK))
    swap = d["lat"].between(*LNG_OK) & d["lng"].between(*LAT_OK)
    bad_pts.append(d[out_].assign(数据=s, 问题=np.where(swap[out_], "经纬度颠倒", "越界")))
bad_pts = pd.concat(bad_pts)
print(f"越界顶点 {len(bad_pts)} 个：")
display(bad_pts[["数据", "name", "seq", "lng", "lat", "问题"]])

# ================= 3. 区域级质检 =================
def self_cross(r):
    """环上不相邻的两条边是否相交（自相交的多边形面积、落点判断都不可信）。"""
    p = r[:-1] if np.allclose(r[0], r[-1]) else r
    n = len(p)
    seg = [(p[i], p[(i + 1) % n]) for i in range(n)]
    cr = lambda o, a, b: (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])
    for i in range(n):
        for j in range(i + 2, n):
            if i == 0 and j == n - 1:
                continue
            (a, b), (c, d) = seg[i], seg[j]
            if cr(a, b, c) * cr(a, b, d) < 0 and cr(c, d, a) * cr(c, d, b) < 0:
                return True
    return False


polys = {s: rings_from_points(d, "name") for s, d in raw.items()}
qc = []
for s, P in polys.items():
    d = raw[s]
    for n, rings in P.items():
        g = d[d["name"] == n]
        nout = int((~(g["lng"].between(*LNG_OK) & g["lat"].between(*LAT_OK))).sum())
        area = sum(abs(np.dot(r[:, 0] * KX, np.roll(r[:, 1] * KY, 1))
                       - np.dot(r[:, 1] * KY, np.roll(r[:, 0] * KX, 1))) / 2 for r in rings)
        cross = nout == 0 and any(self_cross(r) for r in rings)
        keys = ["ring", "seq"] if "ring" in g else ["seq"]
        seq_bad = g["seq"].isna().any() or g.duplicated(keys).any()
        if not seq_bad:                                   # seq 应从 0 起连续编号
            seq_bad = any((gg["seq"].sort_values().to_numpy() != np.arange(len(gg))).any()
                          for _, gg in (g.groupby("ring") if "ring" in g else [(0, g)]))
        unclosed = not all(np.allclose(r[0], r[-1]) for r in rings)
        dup = any((np.abs(np.diff(r, axis=0)).sum(axis=1) == 0).any() for r in rings)   # 相邻顶点重复
        problem = ("顶点越界" if nout else "自相交" if cross else "seq 异常" if seq_bad else
                   "面积≈0" if area < 0.01 else "未闭合" if unclosed else "重复顶点" if dup else "")
        ok = g[g["lng"].between(*LNG_OK) & g["lat"].between(*LAT_OK)]
        qc.append(dict(数据=s, 区域=n, 顶点数=len(g), 越界顶点=nout, 自相交=cross, seq异常=seq_bad,
                       未闭合=unclosed, 重复顶点=dup,
                       面积_平方公里=round(area, 3) if not (nout or cross) else np.nan, 问题=problem,
                       中心lng=ok["lng"].median(), 中心lat=ok["lat"].median()))
qc = pd.DataFrame(qc)
display(qc.pivot(index="区域", columns="数据", values=["问题", "面积_平方公里", "顶点数"]))

# ================= 4. 三份之间是不是坐标系不同：试每种偏移，看剩下多少米 =================
good = qc[qc["问题"] == ""].set_index(["数据", "区域"])
shift_tab = []
for a_, b_ in [("v1", "v2"), ("v1", "json"), ("json", "v2")]:
    common = sorted(set(good.loc[a_].index) & set(good.loc[b_].index)) if a_ in good.index and b_ in good.index else []
    if not common:
        continue
    A, B = good.loc[a_].loc[common], good.loc[b_].loc[common]
    res = {}
    for k, f in SHIFT.items():
        x, y = f(A["中心lng"].to_numpy(), A["中心lat"].to_numpy())
        res[k] = np.median(1000 * np.hypot((x - B["中心lng"].to_numpy()) * KX, (y - B["中心lat"].to_numpy()) * KY))
    best = min(res, key=res.get)
    shift_tab.append(dict(从=a_, 到=b_, 直接相差_米=round(res["无偏移"]), 最可能=best,
                          换算后剩余_米=round(res[best]), 对比区域数=len(common)))
shift_tab = pd.DataFrame(shift_tab)


# ================= 5. 图：行 = 三份数据，列 = 区域；同一列同一比例尺 =================
names = list(dict.fromkeys([n for P in polys.values() for n in P]))
fig, axes = plt.subplots(3, len(names), figsize=(10, 5.6), squeeze=False)
for j, n in enumerate(names):
    q = qc[(qc["区域"] == n) & ~qc["问题"].isin(["顶点越界", "自相交"])]
    ext = [max((r[:, 0].max() - r[:, 0].min()) * KX, (r[:, 1].max() - r[:, 1].min()) * KY)
           for s in q["数据"] for r in polys[s][n]]
    if not ext:                                              # 三份都坏了：用未越界顶点的跨度兜底
        allv = pd.concat([d[d["name"] == n] for d in raw.values()])
        allv = allv[allv["lng"].between(*LNG_OK) & allv["lat"].between(*LAT_OK)]
        ext = [max(np.ptp(allv["lng"]) * KX, np.ptp(allv["lat"]) * KY)]
    win = max(max(ext), 0.3) * 1.35                          # 本列窗口边长（公里）：只看形状，不含版本间的偏移
    for i, s in enumerate(polys):
        ax = axes[i, j]
        row = qc[(qc["数据"] == s) & (qc["区域"] == n)]
        if not row.empty:                                    # 每格以本份边界自身为中心（越界顶点不算）
            g = raw[s][raw[s]["name"] == n]
            g = g[g["lng"].between(*LNG_OK) & g["lat"].between(*LAT_OK)]
            cx, cy = (g["lng"].min() + g["lng"].max()) / 2, (g["lat"].min() + g["lat"].max()) / 2
        if row.empty:
            note, nc = "无此区域", MUTED
        else:
            pb = row["问题"].iat[0]
            c = RED if pb else viz.AREA_COLOR.get(n, MAIN)
            for r in polys[s][n]:
                if not pb:
                    ax.fill(r[:, 0], r[:, 1], facecolor=c, alpha=0.25, edgecolor="none", zorder=2)
                ax.plot(r[:, 0], r[:, 1], color=c, lw=0.9, zorder=4)    # 越界的边会直接冲出格子
            if pb == "顶点越界":
                note = f"{row['越界顶点'].iat[0]} 个顶点越界"
            elif pb:
                note = pb
            else:
                note = f"{row['面积_平方公里'].iat[0]:.2f} 平方公里"
            nc = RED if pb else MUTED
        ax.text(0.5, -0.03, note, transform=ax.transAxes, ha="center", va="top", fontsize=9, color=nc)
        ax.set_xlim(cx - win / 2 / KX, cx + win / 2 / KX)
        ax.set_ylim(cy - win / 2 / KY, cy + win / 2 / KY)
        ax.set_aspect(1 / np.cos(np.radians(lat0)))
        ax.set_xticks([]); ax.set_yticks([])
        for sp_ in ax.spines.values():
            sp_.set_color(GRAY); sp_.set_linewidth(0.5)
        if i == 0:
            ax.set_title(n, fontsize=10, color=INK, fontweight="bold", pad=4)
        if j == 0:
            ax.set_ylabel(s, rotation=0, ha="right", va="center", fontsize=10.5,
                          color=INK, fontweight="bold", labelpad=8)
    scale_bar(axes[-1, j], next(k for k in (5, 2, 1, 0.5, 0.2, 0.1) if k <= win / 3))

# —— 选哪份：问题区域最少；一样少时优先本来就是经纬度的（json 要多一步投影换算）——
nreg = len(names)
score = pd.DataFrame({s: dict(问题区域=int((qc[qc["数据"] == s]["问题"] != "").sum()),
                              缺失区域=nreg - qc[qc["数据"] == s]["区域"].nunique(),
                              需换算=int(s == "json")) for s in polys}).T
score["排序"] = score["问题区域"] + score["缺失区域"]
best = score.sort_values(["排序", "需换算"]).index[0]
display(score)
nb = int(score.loc[best, "排序"])
title = (f"建议用 {best}：{nreg} 个区域边界全部合规" if nb == 0 else
         f"三份都有问题，{best} 最少：{nreg} 个区域中 {nb} 个需修复")
viz.finish(fig, title,
           f"每格以本份边界为中心放大，同列比例尺相同；红色 = 不合规；json 按 {proj} 换算；不合规区域："
           + "，".join(f"{s} {score.loc[s, '排序']} 个" for s in polys),
           SOURCE, "D4_01_区域边界三版对比.png")
plt.show()

In [ ]:
quality.dq_record(con, "DQ-13", "准确性", "商圈边界 v1", "ods_aoi_v1",
          f"太原街商圈面积自相交，未包含太原街地铁站", "不通过",
          evidence="沈阳北站有一个点经度写成 23.45；多边形顶点顺序错乱", 
            decision="弃用 v1，采用 v2")
quality.dq_record(con, "DQ-14", "规范性", "商圈边界 json 版本的坐标单位", "ods_aoi_json",
          f"中街商圈和实际差异太大", "不通过",
          evidence="数值远超经纬度范围，需要换算坐标系才能使用", 
            decision="不使用；需要时向数据提供方确认坐标系")


## 写回维度表dim_cell
把今天的结果整理成一张维度表，每个扇区一行，带上站点编号，坐标，所在区县，所在商圈，所有所有在哪里的人物都可以用他

In [ ]:
aoi_v2 = rings_from_points(v2_pts, "name")
sites['aoi'] = point_in_ploygons(sites["lng"], sites['lat'], aoi_v2)
dim_cell = cell.merge(sites[['site_id', "districts", 'aoi']],
                      on='site_id',
                      how='left')
dim_cell.to_sql("dim_cell", con, if_exists="replace", index=False)

In [ ]:
dim_cell.groupby("aoi")['site_id'].nunique().rename('总站数')

::: IMPORTANT 重要：维度表
- 是什么：DIM维度表。描述事件发生在什么背景下的表，在哪个基站？在哪个区县？在哪个商圈，以及指标的口径。维度表一般行数少，变化慢，被很多事实表反复引用。
- 那为什么需要它呢？这个基站在哪个区县会被停留，出行、客群，平台反复用的，做成一张维度表，所有的表都关联同一份。改一处处处生效，否则每张表你各算一遍，区县的划分迟早会对不上。
- 那企业里面。常见的维度表有日期维、地区维，用户维、商品维，那这就是维度表。
:::

## 基站有多密：站点的间距怎么样
每个站点里最近的站点有多元，各个区县相差多少？

In [ ]:
# dim_cell表读取
dim_cell = pd.read_sql("SELECT site_id, lng, lat, districts FROM dim_cell",con)

In [ ]:
site = dim_cell.groupby('site_id').agg(
    lng = ("lng", 'first'),
    lat = ('lat', 'first'),
    district = ('districts', 'first')
)

### 计算方式1： 分块计算

In [ ]:
%%timeit
lng0, lat0 = 123.4, 41.8                                      # 沈阳市区大约在东经 123.4 度、北纬 41.8 度
kx, ky = 111.320 * np.cos(np.radians(lat0)), 110.574          # 城市尺度下，经纬度可以近似换成平面公里（1 度约多少公里）
xy = np.column_stack([(site["lng"] - lng0) * kx,
        (site["lat"] - lat0) * ky]).astype(np.float32)
nn = np.empty(len(xy), dtype=np.float32)
# 分块算：49,948 个站点两两算距离是 2.5 × 10⁹ 个数，一次性展开要 20 GB 内存，必爆。
# 每次只取 500 个站点对全部站点（500 × 49,948 ≈ 2,500 万个数，约 100 MB），循环 100 轮。
# 讲台提示：这是本课程第一次出现“算不动怎么办”。D5 还会讲另外两招：先筛后算、用 SQL 聚合。
for s0 in range(0, len(xy), 500):
    a = xy[s0:s0 + 500]
    d2 = (a[:, :1] - xy[None, :, 0]) ** 2 + (a[:, 1:] - xy[None, :, 1]) ** 2
    d2[np.arange(len(a)), np.arange(s0, s0 + len(a))] = np.inf   # 排除自己
    nn[s0:s0 + 500] = np.sqrt(d2.min(axis=1))
site["nn_m"] = nn * 1000

area = {}                                                     # 各区县面积（平方公里），用鞋带公式
for name, rings in districts.items():
    area[name] = sum(0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
                     for x, y in (((r[:, 0] - lng0) * kx, (r[:, 1] - lat0) * ky) for r in rings))
by = (site[site["district"] != "市外"].groupby("district")
        .agg(站点数=("nn_m", "size"), 间距中位_米=("nn_m", "median")).sort_values("间距中位_米"))
by["每平方公里站点数"] = by["站点数"] / pd.Series(area)
by.round(1)

### 方法2采用KDTree

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree # pip install scipy

In [ ]:

site = dim_cell.groupby('site_id').agg(
    lng=("lng", 'first'),
    lat=('lat', 'first'),
    district=('districts', 'first')
)
# ============================================================
# 2. 经纬度 → 平面公里（沈阳局部平面近似）
# ============================================================
lng0, lat0 = 123.4, 41.8
kx = 111.320 * np.cos(np.radians(lat0))   # 每度经度 ≈ 多少公里
ky = 110.574                               # 每度纬度 ≈ 多少公里
xy = np.column_stack([
    (site["lng"] - lng0) * kx,
    (site["lat"] - lat0) * ky,
]).astype(np.float64)   # cKDTree 用 float64 更稳
# ============================================================
# 3. KD 树求最近邻（等价于原来的分块暴力 + 排除自己）
# ============================================================
tree = cKDTree(xy)
# k=2：第 0 列是自己（距离 0），第 1 列是真正的最近邻
dist_km, idx = tree.query(xy, k=2)
# 取第 1 列作为最近邻距离（公里），转成米
nn = dist_km[:, 1].astype(np.float32)
site["nn_m"] = nn * 1000
# ============================================================
# 4. 各区县面积（鞋带公式，与原代码一致）
# ============================================================
area = {}
for name, rings in districts.items():
    area[name] = sum(
        0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
        for x, y in (((r[:, 0] - lng0) * kx, (r[:, 1] - lat0) * ky) for r in rings)
    )
# ============================================================
# 5. 分组统计
# ============================================================
by = (site[site["district"] != "市外"]
        .groupby("district")
        .agg(站点数=("nn_m", "size"), 间距中位_米=("nn_m", "median"))
        .sort_values("间距中位_米"))
by["每平方公里站点数"] = by["站点数"] / pd.Series(area)
by = by.round(1).reset_index()

In [ ]:
by

In [ ]:
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.path import Path

from signal_insight import config, viz
from signal_insight.viz import MAIN, GRAY, GRAY_DARK, INK, MUTED

# 沿用 D3_02 的版式和标注逻辑：左图地图 + 引线标注，右图对比；标题区、来源行交给 viz.finish
# 需要 notebook 里已有：by（district / 站点数 / 间距中位_米 / 每平方公里站点数）、
#                      districts（{区县名: [环, ...]}，环为 N×2 的 lng, lat）
SOURCE = f"数据：dim_cell 基站工参 制表：{config.AUTHOR}"


# ---------------- 以下三个函数与 D3_02 相同 ----------------
def scale_bar(ax, km):
    """比例尺：按纬度把公里换成经度差。"""
    (x0_, x1_), (y0_, y1_) = ax.get_xlim(), ax.get_ylim()
    deg = km / (111.32 * np.cos(np.radians(lat0)))
    xs, ys = x0_ + 0.04 * (x1_ - x0_), y0_ + 0.04 * (y1_ - y0_)
    ax.plot([xs, xs + deg], [ys, ys], color=INK, lw=1.2, solid_capstyle="butt")
    for x in (xs, xs + deg):                                         # 两端短竖线
        ax.plot([x, x], [ys, ys + 0.008 * (y1_ - y0_)], color=INK, lw=0.8)
    ax.text(xs + deg / 2, ys + 0.018 * (y1_ - y0_), f"{km} 公里",
            ha="center", va="bottom", fontsize=9, color=INK)


def north_arrow(ax, x=0.95, y=0.95):
    ax.annotate("北", xy=(x, y), xytext=(x, y - 0.08), xycoords="axes fraction",
                ha="center", va="top", fontsize=9, color=INK,
                arrowprops=dict(arrowstyle="-|>", color=INK, lw=0.8, shrinkA=0, shrinkB=0))


def draw_districts(ax, polys, values=None, cmap=None, norm=None, edge="#9AA3AE", lw=0.6):
    """区县边界；给了 values 就按数值填色（分级设色地图）。多边形由调用方传进来，函数里不读库。
    返回所有顶点的范围 (经度最小, 经度最大, 纬度最小, 纬度最大)，用来设视窗。"""
    for name, rings in polys.items():
        fc = cmap(norm(values[name])) if values is not None and name in values else "none"
        for ring in rings:
            ax.fill(ring[:, 0], ring[:, 1], facecolor=fc, edgecolor=edge, lw=lw)
    allpts = np.vstack([r for rings in polys.values() for r in rings])
    return (allpts[:, 0].min(), allpts[:, 0].max(),
            allpts[:, 1].min(), allpts[:, 1].max())


# ---------------- 区县标注：落点 / 分侧 / 拐点（D3_02、D3_03 共用，两张图连线一致）----------------
ELBOW, STUB, TXT_GAP = 0.07, 0.14, 0.02      # 度：拐点离地图外缘、等长短横线、文字离短横线
MAP_H_IN, LABEL_W_IN, HEAD_IN = 7.0, 1.30, 0.50   # 英寸：地图高度、标注文字预留宽度、右图列标题行高。D3_02 / D3_03 必须一致，地图才一样大


def map_frame(bx0, bx1, by0, by1, lat0):
    """由地图外框反推：子图纵向范围、子图宽度（英寸）、横向范围。
    同一份边界 + 同一个 MAP_H_IN → 两张图里的地图一样大，位置也一样。"""
    y0, y1 = by0 - 0.03, by1 + 0.03                                    # 上下各留 0.03°
    dy = (y1 - y0) / MAP_H_IN                                          # 每英寸多少纬度
    dx = dy / np.cos(np.radians(lat0))                                 # 每英寸多少经度
    side_in = (ELBOW + STUB + TXT_GAP) / dx + LABEL_W_IN + 0.05        # 拐点、短横线、文字间隙，再加文字宽度
    w_in = (bx1 - bx0) / dx + 2 * side_in
    xc = (bx0 + bx1) / 2
    return (y0, y1), w_in, (xc - w_in / 2 * dx, xc + w_in / 2 * dx)


def far_districts(by):
    """间距断崖之后的区县（相邻两区比值最大处之后）：D3_03 里用蓝色强调的那一组，D3_02 沿用同一组。"""
    sp_ = by.set_index("district")["间距中位_米"].sort_values()
    r_ = sp_.values[1:] / sp_.values[:-1]
    return list(sp_.index[int(r_.argmax()) + 1:])


def draw_outline(ax, polys, color=None, lw=0.9):
    """全市外缘：只出现一次的边就是外缘（两个区共用的边出现两次，不画），画一圈深色细线。
    区与区之间用白线分开（draw_districts 里 edge="white", lw=0.9）。"""
    n_seen, segs = {}, []
    for rings in polys.values():
        for r in rings:
            for p_, q_ in zip(r, np.roll(r, -1, axis=0)):
                if np.allclose(p_, q_):
                    continue
                key = frozenset((tuple(np.round(p_, 5)), tuple(np.round(q_, 5))))
                n_seen[key] = n_seen.get(key, 0) + 1
                segs.append((key, (p_, q_)))
    ax.add_collection(LineCollection([sg for k_, sg in segs if n_seen[k_] == 1],
                                     colors=color or INK, linewidths=lw, capstyle="round", zorder=3))


def pole_point(rings, lat0):
    """落点：区内离边界最远的点。一定落在区里、不贴边，引线进区的位置比"离中心最近的站点"稳。"""
    r = max(rings, key=len)
    lo, hi = r.min(0), r.max(0)
    gx, gy = np.meshgrid(np.linspace(lo[0], hi[0], 45), np.linspace(lo[1], hi[1], 45))
    g = np.column_stack([gx.ravel(), gy.ravel()])
    g = g[Path(r).contains_points(g)]
    if len(g) == 0:
        return r.mean(axis=0)
    edge = np.vstack([np.linspace(r[i], r[(i + 1) % len(r)], 6, endpoint=False) for i in range(len(r))])
    edge = edge[:: max(1, len(edge) // 400)]
    k = [np.cos(np.radians(lat0)), 1]                                  # 经度差按纬度折成同一长度
    dist = np.sqrt((((g[:, None, :] - edge[None, :, :]) * k) ** 2).sum(-1)).min(1)
    return g[dist.argmax()]


def seg_cross(p1, p2, p3, p4):
    ccw = lambda a, b, c: (c[1] - a[1]) * (b[0] - a[0]) - (b[1] - a[1]) * (c[0] - a[0])
    return ccw(p1, p3, p4) * ccw(p2, p3, p4) < 0 and ccw(p1, p2, p3) * ccw(p1, p2, p4) < 0


def label_districts(ax, polys, texts, bx, ylim, lat0, hot=(), fs=9):
    """左右两列上下等距标注。引线 = 等长短横线 → 拐点 → 直线进区。
    polys {区名: [环]}；texts {区名: 标注文字}；bx=(地图最西, 最东)；ylim=(子图下沿, 上沿)；hot 里的区名用 MAIN 加粗。
    分侧：落点按经度排序，西边一半在左列、东边一半在右列。
    拐点：贴着地图外缘，同一列在同一条竖线上，所以进区的斜线从同一处散开，不会各走各的。
    行位：两列共用同一个上下范围；同一列内引线若交叉就对调位置，直到不交叉。"""
    y0, y1 = ylim
    span = y1 - y0
    anchor = {n: pole_point(r, lat0) for n, r in polys.items()}
    top = min(max(a[1] for a in anchor.values()) + 0.04 * span, y1 - 0.2 * span)   # 顶上留给指北针
    bot = max(min(a[1] for a in anchor.values()) - 0.04 * span, y0 + 0.1 * span)   # 底下留给比例尺
    by_x = sorted(anchor, key=lambda n: anchor[n][0])
    half = (len(by_x) + 1) // 2
    for left, names in ((True, by_x[:half]), (False, by_x[half:])):
        ex = bx[0] - ELBOW if left else bx[1] + ELBOW                  # 拐点
        tx = ex - STUB if left else ex + STUB                          # 文字端
        order = sorted(names, key=lambda n: -anchor[n][1])             # 北 → 南
        ty = dict(zip(order, np.linspace(top, bot, len(order))))
        for _ in range(500):
            moved = False
            for a, b in combinations(order, 2):
                if seg_cross((ex, ty[a]), anchor[a], (ex, ty[b]), anchor[b]):
                    ty[a], ty[b] = ty[b], ty[a]
                    moved = True
            if not moved:
                break
        for n in order:
            lng, lat = anchor[n]
            ax.plot([tx, ex, lng], [ty[n], ty[n], lat], zorder=5, solid_joinstyle="round", **LINE)
            ax.plot(lng, lat, "o", ms=2.2, color=INK, mec="white", mew=0.4, zorder=6)
            ax.text(tx + (-TXT_GAP if left else TXT_GAP), ty[n], texts[n],
                    ha="right" if left else "left", va="center", fontsize=fs, zorder=6,
                    color=MAIN if n in hot else INK, fontweight="bold" if n in hot else "normal")


# ---------------- 数据 ----------------
fmt_n = lambda v: f"{v:,.0f}"
d = by.sort_values("间距中位_米").reset_index(drop=True)             # 间距小 → 大
sp = d.set_index("district")["间距中位_米"]
cnt = d.set_index("district")["站点数"]
dens = d.set_index("district")["每平方公里站点数"]
no_geo = [n for n in d["district"] if n not in districts]
if no_geo:
    print("districts 里找不到：", no_geo, "（地图不画，右图保留）")
polys = {n: [np.asarray(r, float) for r in districts[n]] for n in d["district"] if n in districts}

# 间距断崖：相邻两区比值最大处；断崖之后的区县是本图要强调的一件事（蓝色）
ratio = sp.values[1:] / sp.values[:-1]
cut = int(ratio.argmax())
far = far_districts(by)
r_cnt, r_sp = cnt.max() / cnt.min(), sp.max() / sp.min()

# ---------------- 版面：宽度由地图长宽比反推，标注不会被裁掉 ----------------
NAME_W_IN, BAR1_W_IN, BAR2_W_IN, TXT_W_IN = 0.95, 3.30, 1.55, 0.75

allpts = np.vstack([r for rings in polys.values() for r in rings])
bx0, by0 = allpts.min(0)
bx1, by1 = allpts.max(0)
lat0 = 41.8                                                      # 与 D3_02 相同，两张图的地图比例才一致
(y0, y1), left_w_in, (ax0, ax1) = map_frame(bx0, bx1, by0, by1, lat0)

fig_w = 0.3 + left_w_in + NAME_W_IN + 0.25 + BAR1_W_IN + 0.15 + BAR2_W_IN + 0.10 + TXT_W_IN + 0.2
fig_h = MAP_H_IN + 0.67 + 0.23 * 2 + 0.32 + 0.25                  # + 标题区 + 来源行（viz.finish 的留白）
fig = plt.figure(figsize=(fig_w, fig_h))
# 全部用 GridSpec 排版（不用 add_axes），viz.finish 里的 tight_layout 才不会报警告
gs = fig.add_gridspec(2, 7, hspace=0, wspace=0,
                      height_ratios=[HEAD_IN, MAP_H_IN - HEAD_IN],
                      width_ratios=[left_w_in, NAME_W_IN + 0.25, BAR1_W_IN, 0.15, BAR2_W_IN, 0.10, TXT_W_IN])
a1 = fig.add_subplot(gs[:, 0])

# ============ 左：区县按间距填色（单色浅→深，对数色阶），引线标注 ============
cmap = LinearSegmentedColormap.from_list("间距", ["white", GRAY_DARK])
norm = LogNorm(sp.min() * 0.6, sp.max())                         # 留一点底色，最小值不会纯白
draw_districts(a1, polys, values=sp, cmap=cmap, norm=norm, edge="white", lw=0.9)   # 白线分区：深浅两侧都看得清，小区不糊

draw_outline(a1, polys)                                         # 全市外缘深色线（与 D3_02 共用）
a1.set_xlim(ax0, ax1)
a1.set_ylim(y0, y1)
a1.set_aspect(1 / np.cos(np.radians(lat0)), adjustable="box")   # 视窗已按此比例算好，不改数据范围
a1.set_xticks([]); a1.set_yticks([]); a1.grid(False)
for sp_ in a1.spines.values():
    sp_.set_visible(False)
scale_bar(a1, 20); north_arrow(a1)

# —— 区县标注（落点 / 分侧 / 拐点位置见 label_districts，与 D3_02 共用）——
LINE = dict(color=MUTED, lw=0.45)                                # 引线：细、灰，不和区界抢
label_districts(a1, polys, {n: f"{n} {fmt_n(sp[n])} 米" for n in polys},
                (bx0, bx1), (y0, y1), lat0, hot=far)

# ============ 右：间距条（蓝 = 断崖之后）+ 站点数条 + 密度数字 ============
n = len(d)
yy = np.arange(n)[::-1]                                          # 间距最小排最上


def style(ax, ticks=False):
    ax.set_ylim(-0.6, n - 0.4)
    ax.set_xticks([]); ax.grid(False)
    for s in ("top", "right", "bottom"):
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(MUTED)
    ax.spines["left"].set_linewidth(0.8)
    ax.tick_params(axis="y", length=0)
    if not ticks:
        plt.setp(ax.get_yticklabels(), visible=False)


b1 = fig.add_subplot(gs[1, 2])
b2 = fig.add_subplot(gs[1, 4], sharey=b1)
b3 = fig.add_subplot(gs[1, 6], sharey=b1)

b1.barh(yy, d["间距中位_米"], height=0.62,
        color=[MAIN if x in far else GRAY for x in d["district"]])
b1.set_xlim(0, sp.max() * 1.22)
b1.set_yticks(yy); b1.set_yticklabels(d["district"], fontsize=10, color=INK)
for y, v in zip(yy, d["间距中位_米"]):
    b1.text(v + sp.max() * 0.015, y, fmt_n(v), va="center", fontsize=10, color=INK)
style(b1, ticks=True)

b2.barh(yy, d["站点数"], height=0.62, color=GRAY)
b2.set_xlim(0, cnt.max() * 1.30)
for y, v in zip(yy, d["站点数"]):
    b2.text(v + cnt.max() * 0.02, y, fmt_n(v), va="center", fontsize=10, color=INK)
style(b2)

b3.set_xlim(0, 1)
for y, v in zip(yy, d["每平方公里站点数"]):
    b3.text(1, y, f"{v:.1f}", va="center", ha="right", fontsize=10, color=INK)
style(b3)
b3.axis("off")

for ax_, t, ha, xx in [(b1, "站点间距中位数（米）", "left", 0), (b2, "站点数", "left", 0),
                       (b3, "站点 / 平方公里", "right", 1)]:
    ax_.text(xx, 1.015, t, transform=ax_.transAxes, ha=ha, va="bottom",
             fontsize=10, fontweight="bold", color=INK)

y_cut = yy[cut] - 0.5                                            # 断崖虚线 + 倍数
for ax_ in (b1, b2):
    ax_.axhline(y_cut, color=MUTED, lw=0.5, ls=(0, (4, 3)))
b1.text(sp.max() * 1.20, y_cut + 0.12, f"此处间距跳升 {ratio[cut]:.1f} 倍",
        ha="right", va="bottom", fontsize=9, color=MAIN)

title = f"各区县站点数只差 {r_cnt:.1f} 倍，站点间距却差 {r_sp:.0f} 倍"
viz.finish(fig, title,
           f"左图填色越深间距越大（对数色阶），区名后为站点间距中位数；右图按间距从小到大排序\n"
           f"蓝色 = 间距跳升之后的 {len(far)} 个区县（{fmt_n(sp[far].min())} 米以上，"
           f"站点数 {fmt_n(cnt[far].min())}–{fmt_n(cnt[far].max())}）",
           SOURCE, "D3_03_区县站点间距.png")
plt.show()

- 信令的定位定精度取决于基站密度；和平去站点间距中位数为86米，每平方公里14个，新民市为1385米，相差16倍，未来报告以500米半径识别停留，在中心域适用，在远郊县市偏小，作为方法局限写明